In [73]:
import pandas as pd

In [75]:
df = pd.read_csv('Dataset/Pizza_Sales.csv')
df.head()

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1,1,hawaiian_m,1,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2,2,classic_dlx_m,1,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3,2,five_cheese_l,1,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4,2,ital_supr_l,1,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5,2,mexicana_m,1,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


In [76]:
df.isnull().sum()

pizza_id             0
order_id             0
pizza_name_id        0
quantity             0
order_date           0
order_time           0
unit_price           0
total_price          0
pizza_size           0
pizza_category       0
pizza_ingredients    0
pizza_name           0
dtype: int64

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pizza_id           48620 non-null  int64  
 1   order_id           48620 non-null  int64  
 2   pizza_name_id      48620 non-null  object 
 3   quantity           48620 non-null  int64  
 4   order_date         48620 non-null  object 
 5   order_time         48620 non-null  object 
 6   unit_price         48620 non-null  float64
 7   total_price        48620 non-null  float64
 8   pizza_size         48620 non-null  object 
 9   pizza_category     48620 non-null  object 
 10  pizza_ingredients  48620 non-null  object 
 11  pizza_name         48620 non-null  object 
dtypes: float64(2), int64(3), object(7)
memory usage: 4.5+ MB


### Cleaning

In [82]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed')
df['order_time'] = pd.to_datetime(df['order_time'], format='%H.%M.%S')

ValueError: time data "11:38:36" doesn't match format "%H.%M.%S", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

### Feature Engineering

In [85]:
df['Order_Day'] = df['order_date'].dt.day_name()
df['Order_Month'] = df['order_date'].dt.month_name()
df['Order_Hour'] = df['order_time'].dt.hour

AttributeError: Can only use .dt accessor with datetimelike values

### KPIs

In [ ]:
'''
•	Total Revenue = Sum of total_price
•	Total Pizzas Sold = Sum of quantity
•	Total Orders = Count of unique order_id
•	Average Order Value (AOV) = Total Revenue ÷ Total Orders
•	Average Pizza per Order = Total Pizzas Sold ÷ Total Orders
'''

In [ ]:
total_revenue = sum(df['total_price'])
total_pizza_sold = sum(df['quantity'])
# nunique -> number of uniuque records
total_orders = df['order_id'].nunique()
average_order_value = total_revenue / total_orders
average_pizza_per_order = total_pizza_sold / total_orders

In [ ]:
print(f'Total Revenue: Rs.{total_revenue}')
print(f'Total Pizza Quantity Sold Till Date: {total_pizza_sold}')
print(f'Total Orders: {total_orders}')
# print(f'Average Pizza Order Value: {round(average_order_value, 2)}')
print(f'Average Pizza Order Value: Rs.{average_order_value:,.2f}')
print(f'Average Pizza Per Order: {average_pizza_per_order:,.2f}')

### Analysis

#### Daily Trend - Sales by Day of the Week

In [86]:
weekdays = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
df['Order_Day'] = pd.Categorical(df['Order_Day'], categories=weekdays)

In [88]:
daily_trend = df.groupby('Order_Day', observed=False)['total_price'].sum().reset_index().rename(
    columns = {'total_price': 'Total_Price'}
)
daily_trend['Total_Price'] = daily_trend['Total_Price'].map("Rs.{:,.2f}".format)
daily_trend

,Order_Day,Total_Price
0,Sunday,"Rs.102,116.45"
1,Monday,"Rs.110,471.60"
2,Tuesday,"Rs.115,594.45"
3,Wednesday,"Rs.116,731.20"
4,Thursday,"Rs.121,650.30"
5,Friday,"Rs.129,690.90"
6,Saturday,"Rs.121,605.15"


#### Hourly Trend - Sales by Hour of the Day

In [91]:
hourly_trend = df.groupby('Order_Hour')['total_price'].sum().reset_index().rename(
    columns = {'total_price': 'Total_Price'}
)
hourly_trend['Total_Price'] = hourly_trend['Total_Price'].map("Rs.{:,.2f}".format)
hourly_trend

KeyError: 'Order_Hour'

#### Monthly Trend - Monthly Revenue and Orders

In [ ]:
df['Order_Month_Num'] = df['order_date'].dt.month

In [ ]:
monthly_trend = df.groupby(['Order_Month_Num', 'Order_Month']).agg(
    Total_Price = ('total_price', 'sum'),
    Total_Orders = ('order_id', 'nunique')
).reset_index().sort_values(ascending=False, by='Total_Price')

monthly_trend['Total_Price'] = monthly_trend['Total_Price'].map("Rs.{:,.2f}".format)
monthly_trend[['Order_Month', 'Total_Price', 'Total_Orders']]

% of Sales by Category


In [ ]:
total_revenue = df['total_price'].sum()
cat_sales_pct = df.groupby('pizza_category')['total_price'].sum().reset_index()
cat_sales_pct['sales_pct']=round((cat_sales_pct['total_price']/total_revenue)*100,2)
cat_sales_pct['sales_pct']=cat_sales_pct['sales_pct'].map("{:.2f}".format)
cat_sales_pct['total_price']=cat_sales_pct['total_price'].map("Rs.{:,.2f}%".format)

cat_sales_pct

% of Sales by Size


In [96]:
size_sales_pct = df.groupby('pizza_category')['total_price'].sum().reset_index()

size_sales_pct['sales_pct'] = round((size_sales_pct['total_price'] / total_revenue) * 100, 2)
size_sales_pct['sales_pct']=size_sales_pct['sales_pct'].map("{:.2f}%".format)
size_sales_pct['total_price']=size_sales_pct['total_price'].map("Rs.{:,.2f}".format)
size_sales_pct

,pizza_category,total_price,sales_pct
0,Chicken,"Rs.195,919.50",23.96%
1,Classic,"Rs.220,053.10",26.91%
2,Supreme,"Rs.208,197.00",25.46%
3,Veggie,"Rs.193,690.45",23.68%


#Cross Tab

In [99]:
cross_tab = pd.crosstab(df['pizza_size'], df['pizza_category'])
cross_tab

pizza_category,Chicken,Classic,Supreme,Veggie
pizza_size,,,,
L,4799,3987,4477,5263
M,3811,4045,3977,3552
S,2205,5975,3323,2634
XL,0,544,0,0
XXL,0,28,0,0


#Total pizza sold by pizza category

In [ ]:
cat_pizza_sold = df.groupby('pizza_category')['order_id'].nunique().reset_index().rename(
    columns = {'order_id':'total_pizza_sold'}
)
cat_pizza_sold

### Top 5 best selling pizza


In [104]:
top_pizza = df.groupby('pizza_name')['order_id'].sum().reset_index()
top_pizza





,pizza_name,order_id
0,The Barbecue Chicken Pizza,24871757
1,The Big Meat Pizza,19533329
2,The Brie Carre Pizza,5158921
3,The Calabrese Pizza,10052080
4,The California Chicken Pizza,24753723
5,The Chicken Alfredo Pizza,10517559
6,The Chicken Pesto Pizza,10376438
7,The Classic Deluxe Pizza,26399795
8,The Five Cheese Pizza,14048981
9,The Four Cheese Pizza,19722702


In [106]:
top_pizza = df.groupby('pizza_name').agg(
    num_of_orders=('order_id', 'count'),
    total_sales=('total_price', 'sum'),
    quantity=('quantity', 'sum')   
).reset_index()

# Top 5 by number of orders
top5_by_orders = top_pizza.sort_values('num_of_orders', ascending=False,ignore_index=True).head(5)
top5_by_orders

# Top 5 by total sales
top5_by_sales = top_pizza.sort_values('total_sales', ascending=False, ignore_index=True).head(5)
top5_by_sales

# Top 5 by quantity
top5_by_quantity = top_pizza.sort_values('quantity', ascending=False, ignore_index=True).head(5)
top5_by_quantity


,pizza_name,num_of_orders,total_sales,quantity
0,The Classic Deluxe Pizza,2416,38180.50,2453
1,The Barbecue Chicken Pizza,2372,42768.00,2432
2,The Hawaiian Pizza,2370,32273.25,2422
3,The Pepperoni Pizza,2369,30161.75,2418
4,The Thai Chicken Pizza,2315,43434.25,2371


In [108]:
top_pizza = df.groupby('pizza_name').agg(
    num_of_orders=('order_id', 'count'),
    total_sales=('total_price', 'sum'),
    quantity=('quantity', 'sum')   
).reset_index()

# Least 5 by number of orders
top5_by_orders = top_pizza.sort_values('num_of_orders', ascending=True,ignore_index=True).head(5)
top5_by_orders

# Least 5 by total sales
top5_by_sales = top_pizza.sort_values('total_sales', ascending=True, ignore_index=True).head(5)
top5_by_sales

# Least 5 by quantity
top5_by_quantity = top_pizza.sort_values('quantity', ascending=True, ignore_index=True).head(5)
top5_by_quantity


,pizza_name,num_of_orders,total_sales,quantity
0,The Brie Carre Pizza,480,11588.50,490
1,The Mediterranean Pizza,923,15360.50,934
2,The Calabrese Pizza,927,15934.25,937
3,The Spinach Supreme Pizza,940,15277.75,950
4,The Soppressata Pizza,957,16425.75,961
